In [73]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import jieba
from gensim.models import KeyedVectors

In [74]:
kv=KeyedVectors.load_word2vec_format('data/sgns.weibo.word.bz2')


In [75]:
print(kv.vector_size)

300


In [76]:
kv.vectors.shape

(195202, 300)

In [77]:
# 查看某个词的词向量
id=kv.key_to_index['地铁']
print(id)
vector=kv.vectors[id]
print(vector.shape)

1143
(300,)


In [78]:
print(kv['河南'])

[ 0.413979 -0.141125  0.221013 -0.456477 -0.305955  0.520065  0.035824
 -0.402475 -0.266599  0.248628  0.283796  0.275114 -0.242932  0.11156
  0.292311  0.617277 -0.186553  0.160774  0.23333  -0.203058  0.024398
  0.268656  0.144249  0.443972  0.102098  0.269671 -0.090864 -0.555971
  0.053002  0.018031  0.404913  0.026849 -0.184152  0.479678  0.111152
 -0.090827 -0.224237 -0.095617 -0.091896  0.134677 -0.412746  0.040212
  0.60851   0.009644  0.268446 -0.377374 -0.15255  -0.154629 -0.052627
  0.096748 -0.423573 -0.191556 -0.094728  0.276766  0.163386  0.804496
  0.131839  0.058252  0.17452   0.255434 -0.362472  0.001015  0.391725
  0.447637 -0.051783  0.140423 -0.15065  -0.756959 -0.02875   0.176741
  0.220562 -0.31519  -0.374408 -0.315362 -0.549119  0.104749  0.163208
  0.30859   0.354518  0.267746 -0.02257   0.457168  0.105569  0.245613
 -0.203871  0.049795  0.024358 -0.173262 -0.151903  0.083175 -0.087717
  0.49141   0.229067 -0.029952 -0.29149   0.323356  0.418935 -0.678067
  0.108

# API调用

In [79]:
# 相似度 与接近于1越相关
print(kv.similarity('河南','广东'))
print(kv.similarity('河南','河南大学'))
print(kv.similarity('河南','郑州'))
print(kv.similarity('河南','河南'))
print(kv.similarity('地铁','公交'))
print(kv.similarity('宇通','公交'))



0.33814204
0.41186655
0.55406344
1.0
0.65458214
0.40268204


In [80]:
# 近义词
print(kv.similar_by_word('地铁'))

[('二号线', 0.6998022198677063), ('四号线', 0.6872348785400391), ('北京地铁', 0.6863653659820557), ('一号线', 0.6666116118431091), ('地铁站', 0.659015417098999), ('公交', 0.654582142829895), ('五号线', 0.6538913249969482), ('坐地铁', 0.6435028314590454), ('军博', 0.6391095519065857), ('八通线', 0.6323882341384888)]


In [81]:
# 正负相关的近义词
similar_words=kv.most_similar(['地铁'])
print(similar_words)

[('二号线', 0.6998022198677063), ('四号线', 0.6872348785400391), ('北京地铁', 0.6863653659820557), ('一号线', 0.6666116118431091), ('地铁站', 0.659015417098999), ('公交', 0.654582142829895), ('五号线', 0.6538913249969482), ('坐地铁', 0.6435028314590454), ('军博', 0.6391095519065857), ('八通线', 0.6323882341384888)]


In [82]:
# 正负相关的近义词
similar_words=kv.most_similar(['地铁','公交'])
print(similar_words)

[('二号线', 0.7100438475608826), ('四号线', 0.7079388499259949), ('公交车', 0.701663613319397), ('北京地铁', 0.6874836087226868), ('换乘', 0.6760745644569397), ('坐地铁', 0.6700716614723206), ('五号线', 0.667486846446991), ('快轨', 0.661508321762085), ('惠新西街南口', 0.6552888751029968), ('换乘站', 0.6544797420501709)]


In [83]:
# 对比填词
result=kv.most_similar(['男人','女孩'],['女人'],topn=5)
print(result)

[('男孩', 0.5668050050735474), ('小伙子', 0.48005053400993347), ('小伙', 0.4718535244464874), ('小男孩', 0.46238160133361816), ('男孩子', 0.45248618721961975)]


# 自行训练

In [84]:
from gensim.models import Word2Vec
import jieba

In [85]:
text1='我每天乘坐地铁上班'
text2='我每天乘坐公交上班'

In [86]:
sentences=[jieba.lcut(text) for text in [text1,text2]]
print(sentences)

[['我', '每天', '乘坐', '地铁', '上班'], ['我', '每天', '乘坐', '公交', '上班']]


In [87]:
# 定义
model=Word2Vec(
    sentences=sentences,
    vector_size=10,
    window=2,
    min_count=1,
    workers=4,
    sg=1,

)

In [88]:
print(model)

Word2Vec<vocab=6, vector_size=10, alpha=0.025>


In [89]:
print(model.wv['地铁'])

[-0.08157917  0.04495798 -0.04137076  0.00824536  0.08498619 -0.04462177
  0.045175   -0.0678696  -0.03548489  0.09398508]


# 基于真实语料库

In [95]:
import pandas as pd

In [96]:
df=pd.read_csv('./data/online_shopping_10_cats.csv')
df.dropna(inplace=True)
print(df.shape)

(62773, 3)


In [97]:
sentences=[jieba.lcut(text) for text in df ['review']]

In [98]:
model=Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=2,
    min_count=1,
    workers=4,
    sg=1,
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
